# Broadband Continuous Measured and Synthetic Signal Processing with STFT-Based Propagation

This notebook demonstrates broadband passive-sonar signal generation and reception using the `BroadbandPassiveSonarArraySimulator`. It follows a single moving target and a towed array through source generation, frequency-domain propagation, array reception, and signal analysis. It compares a synthetic source signal with a measured ship noise signal. The measured signal is imported from a WAV file which is available from [Sanct Sounds](https://sanctsound.ioos.us/sounds.html#Vessels). 

**Background**
- Broadband ship noise is not well represented by a handful of discrete tones; realistic signals span a wider band and evolve continuously in time.
- For moving-source, moving-array problems, propagation must preserve phase and timing well enough to reconstruct received sensor data after frequency-domain processing.
- In passive sonar analysis, spectrograms and spectra are only interpretable if FFT scaling and PSD conventions are handled correctly.

**Key Concepts**
- STFT-based frequency-domain propagation and overlap-add reconstruction.
- Broadband source modelling with time-varying transfer functions.
- Correct amplitude interpretation in FFT and spectrogram-based plots.


## Setup and Reproducibility


In [ ]:
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from stonesoup.models.transition.linear import (
    CombinedLinearGaussianTransitionModel,
    ConstantVelocity,
)
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState

from bluepebble.models.environment import FlatBathymetry, Constant
from bluepebble.models.propagation import CylindricalAcousticPropagationModel
from bluepebble.platform import TowedArrayPlatform
from bluepebble.plotter import plot_spectrogram, plot_world
from bluepebble.signal.anthropogenic import BroadbandMeasuredSignal, BroadbandShipSignal
from bluepebble.simulator import BroadbandPassiveSonarArraySimulator

# Set random seed for reproducibility
np.random.seed(1999)

## Simulation Parameters

Configure the simulation timing: 60 seconds total duration with 2-second timesteps (30 steps).

In [138]:
# Simulation parameters
SIM_RATE = 2.0
SIM_LENGTH_S = 100.0
SIM_PARAMS = {
    "start_time": datetime.now().replace(hour=0, minute=0, second=0, microsecond=0),
    "time_interval": timedelta(seconds=SIM_RATE),
    "num_steps": int(SIM_LENGTH_S / SIM_RATE),
}

total_duration_s = SIM_PARAMS["num_steps"] * SIM_PARAMS["time_interval"].total_seconds()

print("=== Broadband Continuous Signal Processing Demo ===")
print(f"Total simulation duration: {total_duration_s} s")
print(f"Number of timesteps: {SIM_PARAMS['num_steps']}")
print(f"Timestep interval: {SIM_PARAMS['time_interval'].total_seconds()} s")

=== Broadband Continuous Signal Processing Demo ===
Total simulation duration: 100.0 s
Number of timesteps: 50
Timestep interval: 2.0 s


## Platform and Array Parameters

The ship platform remains stationary while the towed array (100 sensors, 200m cable, 1m spacing) is deployed at 200m depth.

In [139]:
SHIP_PARAMS = {
    "start_vector": np.array([0, 0, 0, 0, -10.0, 0]),  # Stationary ship
    "position_mapping": [0, 2, 4],
    "velocity_mapping": [1, 3, 5],
    "transition_model": CombinedLinearGaussianTransitionModel(
        [ConstantVelocity(0), ConstantVelocity(0), ConstantVelocity(0)]
    ),
}

ARRAY_PARAMS = {
    "num_sensors": 100,
    "tow_cable_length": 200.0,
    "sensor_spacing": 1.0,
    "array_depth": -10.0,
}

print("Platform: Stationary ship at surface")
print(
    f"Array: {ARRAY_PARAMS['num_sensors']} sensors, "
    f"{ARRAY_PARAMS['tow_cable_length']}m cable, "
    f"{ARRAY_PARAMS['sensor_spacing']}m spacing"
)
print(f"Array depth: {ARRAY_PARAMS['array_depth']} m")

Platform: Stationary ship at surface
Array: 100 sensors, 200.0m cable, 1.0m spacing
Array depth: -10.0 m


## Target Parameters

Here the components of the synthetic signal are defined. The acoustic signature consists of 4 tonal components at different frequencies (60, 85, 120, 200 Hz) with amplitudes ranging from 75-100 dB re 1 µPa.

In [140]:
TARGET_PARAMS = {
    "start_vector": np.array([0, 0, 500, 5.0, -10.0, 0]),  # Moving target
    "position_mapping": [0, 2, 4],
    "velocity_mapping": [1, 3, 5],
    "transition_model": CombinedLinearGaussianTransitionModel(
        [ConstantVelocity(0.001), ConstantVelocity(0.001), ConstantVelocity(0)]
    ),
    "amplitudes_upa": 10 ** (np.array([100.0, 90.0, 100.0, 85.0]) / 20),
    "frequencies_hz": np.array([20.0, 140.0, 200.0, 500.0]),
    "phases_rad": np.random.uniform(0, 2 * np.pi, 4),
    "tonal_bandwidth_hz": 10.0,
    "noise_amplitude_upa": 10 ** (80 / 20),
    "noise_spectral_exponent": -1.0,
}

# Convert amplitudes to dB for display
amplitudes_db = 20 * np.log10(TARGET_PARAMS["amplitudes_upa"])

print(f"Target initial position: {TARGET_PARAMS['start_vector'][2]:.0f} m range")
print(f"Target depth: {TARGET_PARAMS['start_vector'][4]:.0f} m")
print(f"Tonal frequencies: {TARGET_PARAMS['frequencies_hz']} Hz")
print(f"Tonal amplitudes: {amplitudes_db} dB re 1 µPa")

Target initial position: 500 m range
Target depth: -10 m
Tonal frequencies: [ 20. 140. 200. 500.] Hz
Tonal amplitudes: [100.  90. 100.  85.] dB re 1 µPa


## Signal and Propagation Parameters

In [141]:
SIGNAL_PARAMS = {
    "duration_s": total_duration_s,  # Long continuous signal
    "sampling_rate_hz": 600.0,
    "frame_len": 600,  # STFT frame length
    "hop_factor": 4,  
    "fade_in_ms": 100.0,
    "fade_out_ms": 100.0,  
}

# Sensor to analyze
SENSOR_TO_ANALYZE = ARRAY_PARAMS["num_sensors"] // 2

## Platform Generation

Initialize the towed array platform and simulate motion through timesteps.

In [142]:
print("Creating platform with vertical motion...")
initial_state = GroundTruthState(SHIP_PARAMS["start_vector"], timestamp=SIM_PARAMS["start_time"])

platform = TowedArrayPlatform(
    states=[initial_state],
    position_mapping=SHIP_PARAMS["position_mapping"],
    velocity_mapping=SHIP_PARAMS["velocity_mapping"],
    transition_models=[SHIP_PARAMS["transition_model"]],
    transition_times=[timedelta(seconds=total_duration_s)],
    num_sensors=ARRAY_PARAMS["num_sensors"],
    cable_length_m=ARRAY_PARAMS["tow_cable_length"],
    sensor_spacing_m=ARRAY_PARAMS["sensor_spacing"],
    array_depth_m=ARRAY_PARAMS["array_depth"],
)

for i in range(1, SIM_PARAMS["num_steps"]):
    new_time = SIM_PARAMS["start_time"] + i * SIM_PARAMS["time_interval"]
    platform.move(new_time)

Creating platform with vertical motion...


## Target Trajectory

Generate the target ground truth path with acoustic metadata.

In [143]:
target_states = [
    GroundTruthState(
        TARGET_PARAMS["start_vector"],
        timestamp=SIM_PARAMS["start_time"],
        metadata={
            "amplitudes_upa": TARGET_PARAMS["amplitudes_upa"],
            "frequencies_hz": TARGET_PARAMS["frequencies_hz"],
            "phases_rad": TARGET_PARAMS["phases_rad"],
            "position_mapping": TARGET_PARAMS["position_mapping"],
            "velocity_mapping": TARGET_PARAMS["velocity_mapping"],
        },
    )
]

transition_model = TARGET_PARAMS["transition_model"]
for i in range(1, SIM_PARAMS["num_steps"]):
    new_time = SIM_PARAMS["start_time"] + i * SIM_PARAMS["time_interval"]
    time_interval = new_time - target_states[-1].timestamp
    new_state_vector = transition_model.function(
        target_states[-1], noise=False, time_interval=time_interval
    )
    new_state = GroundTruthState(
        new_state_vector,
        timestamp=new_time,
        metadata=target_states[-1].metadata,
    )
    target_states.append(new_state)

target_ground_truth = GroundTruthPath(target_states)

# Use plotter helper for world view
fig_world = plot_world(truths=[target_ground_truth], platform=platform)
fig_world.update_layout(title="World Picture")
fig_world.show()

## Broadband Signal Models

Define both synthetic and measured broadband source models so they can be compared in the same notebook. The measured signal is imported from a WAV file which is available from [Sanct Sounds](https://sanctsound.ioos.us/sounds.html#Vessels). The synthetic signal consists of 4 tonal components at different frequencies (60, 85, 120, 200 Hz) with amplitudes ranging from 75-100 dB re 1 µPa.

In [144]:
synthetic_signal_model = BroadbandShipSignal(
    duration_s=SIGNAL_PARAMS["duration_s"],
    sampling_rate_hz=SIGNAL_PARAMS["sampling_rate_hz"],
    frame_len=SIGNAL_PARAMS["frame_len"],
    hop_factor=SIGNAL_PARAMS["hop_factor"],
    tonal_bandwidth_hz=TARGET_PARAMS["tonal_bandwidth_hz"],
    noise_amplitude_upa=TARGET_PARAMS["noise_amplitude_upa"],
    noise_spectral_exponent=TARGET_PARAMS["noise_spectral_exponent"],
    noise_freq_range_hz=(0.0, SIGNAL_PARAMS["sampling_rate_hz"] / 2),
    tonal_noise_is_constant=True,
    noise_is_constant=True,
)

wav_name = "SanctSound_CI05_03_largeship_20190925T135956Z.wav"
measured_wav_path = Path("measured_data") / wav_name    # for notebooks
# data_dir = Path(__file__).resolve().parent / "measured_data"  # for scripts
# measured_wav_path = data_dir / wav_name

measured_signal_model = BroadbandMeasuredSignal(
    duration_s=SIGNAL_PARAMS["duration_s"],
    sampling_rate_hz=SIGNAL_PARAMS["sampling_rate_hz"],
    frame_len=SIGNAL_PARAMS["frame_len"],
    hop_factor=SIGNAL_PARAMS["hop_factor"],
    wav_path=str(measured_wav_path),
    segment_start_s=0.0,
    segment_duration_s=30.0,
    duration_match_mode="tile",
    level_db_re_1upa=85.0,
)

## Propagation Model and Simulator

Set up the **rtrs** (Range-dependent Two-dimensional Ray Simulation) acoustic propagation model and initialize the broadband passive sonar simulator.

In [ ]:
ssp = Constant(speed=1500.0)
bathymetry = FlatBathymetry(depth=-100.0)

prop_model = CylindricalAcousticPropagationModel(
    attenuation_factor=5.0,
    ssp=ssp,
)

synthetic_simulator = BroadbandPassiveSonarArraySimulator(
    platform=platform,
    propagation_model=prop_model,
    signal_models=[synthetic_signal_model],
    noise_model=None,
    beamformer=None,
    steering_calculator=None,
    ground_truth_paths=[target_ground_truth],
    fade_in_ms=SIGNAL_PARAMS["fade_in_ms"],
)

measured_simulator = BroadbandPassiveSonarArraySimulator(
    platform=platform,
    propagation_model=prop_model,
    signal_models=[measured_signal_model],
    noise_model=None,
    beamformer=None,
    steering_calculator=None,
    ground_truth_paths=[target_ground_truth],
    fade_in_ms=SIGNAL_PARAMS["fade_in_ms"],
)

## Run Simulation and Collect Sensor Data

Execute the simulation loop to generate sensor data at each timestep. The simulator processes signals in the frequency domain using STFT, applies propagation transfer functions, and reconstructs time-domain signals via overlap-add.

**Note**: This step may take several minutes depending on system performance.

In [146]:
def run_continuous_simulation(simulator_obj):
    all_sensor_signals = []

    for _, sensor_data_set in simulator_obj.sensor_data_gen():
        sensor_data = next(iter(sensor_data_set))
        all_sensor_signals.append(sensor_data.raw_signals)

    all_sensor_signals_array = np.concatenate(all_sensor_signals, axis=1)
    return {
        "all_sensor_signals_array": all_sensor_signals_array,
        "continuous_signal": all_sensor_signals_array[SENSOR_TO_ANALYZE, :],
        "continuous_signal_first": all_sensor_signals_array[0, :],
        "continuous_signal_last": all_sensor_signals_array[-1, :],
    }

print("Running synthetic simulation...")
synthetic_results = run_continuous_simulation(synthetic_simulator)

print("Running measured simulation...")
measured_results = run_continuous_simulation(measured_simulator)

# Keep existing downstream analysis on synthetic by default
all_sensor_signals_array = synthetic_results["all_sensor_signals_array"]
continuous_signal = synthetic_results["continuous_signal"]
continuous_signal_first = synthetic_results["continuous_signal_first"]
continuous_signal_last = synthetic_results["continuous_signal_last"]

# Additional measured outputs for comparison
all_sensor_signals_array_measured = measured_results["all_sensor_signals_array"]
continuous_signal_measured = measured_results["continuous_signal"]
continuous_signal_first_measured = measured_results["continuous_signal_first"]
continuous_signal_last_measured = measured_results["continuous_signal_last"]

time_axis = np.arange(continuous_signal.shape[0]) / SIGNAL_PARAMS["sampling_rate_hz"]
time_axis_measured = np.arange(continuous_signal_measured.shape[0]) / SIGNAL_PARAMS["sampling_rate_hz"]

print(f"Synthetic duration: {len(continuous_signal) / SIGNAL_PARAMS['sampling_rate_hz']:.1f} s")
print(f"Measured duration: {len(continuous_signal_measured) / SIGNAL_PARAMS['sampling_rate_hz']:.1f} s")

Running synthetic simulation...
Running measured simulation...
Synthetic duration: 99.5 s
Measured duration: 99.5 s


## Retrieve Source Signal

Extract the source signal from the signal model cache

In [147]:
try:
    source_signal_synthetic = synthetic_signal_model.get_source_signal()
except RuntimeError:
    print("Computing STFT to generate synthetic source signal...")
    synthetic_signal_model.compute_stft(target_states[0])
    source_signal_synthetic = synthetic_signal_model.get_source_signal()

try:
    source_signal_measured = measured_signal_model.get_source_signal()
except RuntimeError:
    print("Computing STFT to generate measured source signal...")
    measured_signal_model.compute_stft(target_states[0])
    source_signal_measured = measured_signal_model.get_source_signal()

## Results

Bellow results are shown to compare the two methods.

### Source Signal Spectrograms

In [148]:
synthetic_source_real = np.real(source_signal_synthetic)
measured_source_real = np.real(source_signal_measured)


# Synthetic Source Spectrogram 
fig_synth_rx_spec = plot_spectrogram(
    synthetic_source_real,
    int(SIGNAL_PARAMS["sampling_rate_hz"]),
    n_fft=500,
    hop_length=250,
    y_lim=(0, SIGNAL_PARAMS["sampling_rate_hz"] / 2),
    yaxis_format="hz",
    figsize=(9, 4),
)
fig_synth_rx_spec.update_layout(title="Spectrogram - Synthetic Source signal Signal")
fig_synth_rx_spec.show()

# Measured Source Spectrogram
fig_meas_rx_spec = plot_spectrogram(
    measured_source_real,
    int(SIGNAL_PARAMS["sampling_rate_hz"]),
    n_fft=500,
    hop_length=250,
    y_lim=(0, SIGNAL_PARAMS["sampling_rate_hz"] / 2),
    yaxis_format="hz",
    figsize=(9, 4),
)
fig_meas_rx_spec.update_layout(title="Spectrogram - Measured Source signal Signal")
fig_meas_rx_spec.show()


### Received Signal Spectrograms

In [149]:
synthetic_received_real = np.real(synthetic_results["continuous_signal"])
measured_received_real = np.real(measured_results["continuous_signal"])

# Synthetic received Spectrogram 
fig_synth_rx_spec = plot_spectrogram(
    synthetic_received_real,
    int(SIGNAL_PARAMS["sampling_rate_hz"]),
    n_fft=500,
    hop_length=250,
    y_lim=(0, SIGNAL_PARAMS["sampling_rate_hz"] / 2),
    yaxis_format="hz",
    figsize=(9, 4),
)
fig_synth_rx_spec.update_layout(title="Spectrogram - Synthetic Received signal Signal (sensor {})".format(SENSOR_TO_ANALYZE))
fig_synth_rx_spec.show()

# Measured received Spectrogram
fig_meas_rx_spec = plot_spectrogram(
    measured_received_real,
    int(SIGNAL_PARAMS["sampling_rate_hz"]),
    n_fft=500,
    hop_length=250,
    y_lim=(0, SIGNAL_PARAMS["sampling_rate_hz"] / 2),
    yaxis_format="hz",
    figsize=(9, 4),
)
fig_meas_rx_spec.update_layout(title="Spectrogram - Measured Received signal Signal (sensor {})".format(SENSOR_TO_ANALYZE))
fig_meas_rx_spec.show()



### Frequency Spectrums

In [150]:
# Measured spectrums

freq_axis_synthetic_source = np.fft.rfftfreq(
    len(synthetic_source_real), 1 / SIGNAL_PARAMS["sampling_rate_hz"]
)
freq_axis_synthetic_received = np.fft.rfftfreq(
    len(synthetic_received_real), 1 / SIGNAL_PARAMS["sampling_rate_hz"]
)
spectrum_source_synthetic = np.abs(np.fft.rfft(synthetic_source_real)) * (2.0 / len(synthetic_source_real))
spectrum_received_synthetic = np.abs(np.fft.rfft(synthetic_received_real)) * (2.0 / len(synthetic_received_real))
spectrum_source_synthetic_db = 20 * np.log10(spectrum_source_synthetic + 1e-10)
spectrum_received_synthetic_db = 20 * np.log10(spectrum_received_synthetic + 1e-10)

fig_meas_freq = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    subplot_titles=(
        "synthetic Source Signal - Frequency Spectrum",
        f"synthetic Received Signal - Frequency Spectrum (Sensor {SENSOR_TO_ANALYZE})",
    ),
)
fig_meas_freq.add_trace(
    go.Scatter(
        x=freq_axis_synthetic_source,
        y=spectrum_source_synthetic_db,
        mode="lines",
        line=dict(width=1, color="steelblue"),
        showlegend=False,
    ),
    row=1,
    col=1,
)
fig_meas_freq.add_trace(
    go.Scatter(
        x=freq_axis_synthetic_received,
        y=spectrum_received_synthetic_db,
        mode="lines",
        line=dict(width=1, color="darkorange"),
        showlegend=False,
    ),
    row=2,
    col=1,
)
fig_meas_freq.update_yaxes(title_text="Magnitude (dB re 1 µPa)", row=1, col=1)
fig_meas_freq.update_yaxes(title_text="Magnitude (dB re 1 µPa)", row=2, col=1)
fig_meas_freq.update_xaxes(
    title_text="Frequency (Hz)",
    range=[0, SIGNAL_PARAMS["sampling_rate_hz"] / 2],
    row=2,
    col=1,
)
fig_meas_freq.update_layout(template="plotly_white", height=760, width=900)
fig_meas_freq.show()


# Measured spectrums
freq_axis_measured_source = np.fft.rfftfreq(
    len(measured_source_real), 1 / SIGNAL_PARAMS["sampling_rate_hz"]
)
freq_axis_measured_received = np.fft.rfftfreq(
    len(measured_received_real), 1 / SIGNAL_PARAMS["sampling_rate_hz"]
)
spectrum_source_measured = np.abs(np.fft.rfft(measured_source_real)) * (2.0 / len(measured_source_real))
spectrum_received_measured = np.abs(np.fft.rfft(measured_received_real)) * (2.0 / len(measured_received_real))
spectrum_source_measured_db = 20 * np.log10(spectrum_source_measured + 1e-10)
spectrum_received_measured_db = 20 * np.log10(spectrum_received_measured + 1e-10)

fig_meas_freq = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    subplot_titles=(
        "Measured Source Signal - Frequency Spectrum",
        f"Measured Received Signal - Frequency Spectrum (Sensor {SENSOR_TO_ANALYZE})",
    ),
)
fig_meas_freq.add_trace(
    go.Scatter(
        x=freq_axis_measured_source,
        y=spectrum_source_measured_db,
        mode="lines",
        line=dict(width=1, color="steelblue"),
        showlegend=False,
    ),
    row=1,
    col=1,
)
fig_meas_freq.add_trace(
    go.Scatter(
        x=freq_axis_measured_received,
        y=spectrum_received_measured_db,
        mode="lines",
        line=dict(width=1, color="darkorange"),
        showlegend=False,
    ),
    row=2,
    col=1,
)
fig_meas_freq.update_yaxes(title_text="Magnitude (dB re 1 µPa)", row=1, col=1)
fig_meas_freq.update_yaxes(title_text="Magnitude (dB re 1 µPa)", row=2, col=1)
fig_meas_freq.update_xaxes(
    title_text="Frequency (Hz)",
    range=[0, SIGNAL_PARAMS["sampling_rate_hz"] / 2],
    row=2,
    col=1,
)
fig_meas_freq.update_layout(template="plotly_white", height=760, width=900)
fig_meas_freq.show()

### Timeseries Plots

In [151]:

# --- Time-domain source waveform comparison ---
fig_time_source = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    subplot_titles=(
        "Synthetic Source Time Series",
        "Measured Source Time Series",
    ),
)

fig_time_source.add_trace(
    go.Scatter(
        x=np.arange(len(synthetic_source_real)) / SIGNAL_PARAMS["sampling_rate_hz"],
        y=synthetic_source_real,
        mode="lines",
        line=dict(width=1, color="steelblue"),
        name="Synthetic Source",
        showlegend=False,
    ),
    row=1,
    col=1,
)
fig_time_source.add_trace(
    go.Scatter(
        x=np.arange(len(measured_source_real)) / SIGNAL_PARAMS["sampling_rate_hz"],
        y=measured_source_real,
        mode="lines",
        line=dict(width=1, color="darkorange"),
        name="Measured Source",
        showlegend=False,
    ),
    row=2,
    col=1,
)

fig_time_source.update_yaxes(title_text="Amplitude (µPa)", row=1, col=1)
fig_time_source.update_yaxes(title_text="Amplitude (µPa)", row=2, col=1)
fig_time_source.update_xaxes(title_text="Time (s)", row=2, col=1)
fig_time_source.update_layout(template="plotly_white", height=760, width=900)
fig_time_source.show()

# --- Time-domain received waveform comparison ---
fig_time_received = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    subplot_titles=(
        f"Synthetic Received Time Series (Sensor {SENSOR_TO_ANALYZE})",
        f"Measured Received Time Series (Sensor {SENSOR_TO_ANALYZE})",
    ),
)

fig_time_received.add_trace(
    go.Scatter(
        x=time_axis,
        y=synthetic_received_real,
        mode="lines",
        line=dict(width=1, color="steelblue"),
        name="Synthetic Received",
        showlegend=False,
    ),
    row=1,
    col=1,
)
fig_time_received.add_trace(
    go.Scatter(
        x=time_axis_measured,
        y=measured_received_real,
        mode="lines",
        line=dict(width=1, color="darkorange"),
        name="Measured Received",
        showlegend=False,
    ),
    row=2,
    col=1,
)

fig_time_received.update_yaxes(title_text="Amplitude (µPa)", row=1, col=1)
fig_time_received.update_yaxes(title_text="Amplitude (µPa)", row=2, col=1)
fig_time_received.update_xaxes(title_text="Time (s)", row=2, col=1)
fig_time_received.update_layout(template="plotly_white", height=760, width=900)
fig_time_received.show()



## Save Output WAV files



In [152]:
import scipy.io.wavfile as wavfile

def to_wav_float32(signal: np.ndarray) -> np.ndarray:
    """Convert signal to mono float32 and normalize to avoid clipping in WAV."""
    signal_real = np.real(signal).astype(np.float32)
    peak = np.max(np.abs(signal_real))
    if peak > 0:
        signal_real = 0.99 * signal_real / peak
    return signal_real


wav_output_dir = Path("measured_data") / "exported_signals" # for notebooks
# wav_output_dir = Path(__file__).resolve().parent / "measured_data" / "exported_signals"   # for scripts
wav_output_dir.mkdir(parents=True, exist_ok=True)

syn_source_wav_path = wav_output_dir / "synthetic_source_signal_used.wav"
syn_received_wav_path = wav_output_dir / f"synthetic_received_signal_sensor_{SENSOR_TO_ANALYZE}.wav"
mes_source_wav_path = wav_output_dir / "measured_source_signal_used.wav"
mes_received_wav_path = wav_output_dir / f"measured_received_signal_sensor_{SENSOR_TO_ANALYZE}.wav"

wav_rate_hz = int(SIGNAL_PARAMS["sampling_rate_hz"])
wavfile.write(syn_source_wav_path, wav_rate_hz, to_wav_float32(source_signal_synthetic))
wavfile.write(syn_received_wav_path, wav_rate_hz, to_wav_float32(continuous_signal))
wavfile.write(mes_source_wav_path, wav_rate_hz, to_wav_float32(source_signal_measured))
wavfile.write(mes_received_wav_path, wav_rate_hz, to_wav_float32(continuous_signal_measured))